In [0]:
-- ============================================================================
-- 00_setup_catalogo.sql
-- Proyecto: Combustibles en Honduras - Arquitectura Medallion en Databricks
-- Autor:    Norman Sabillon
-- Fecha:    2026-05-12
--
-- COMO USAR ESTE ARCHIVO:
--   1. Abri Databricks -> menu izquierdo "SQL editor".
--   2. New query.
--   3. Copia y pega TODO este archivo.
--   4. Asegurate de tener un SQL Warehouse encendido (arriba a la derecha).
--   5. Click "Run all" (o boton Play).
--   6. Verifica los resultados de los SELECT al final.
-- ============================================================================


-- ----------------------------------------------------------------------------
-- PASO 1: Crear el CATALOGO principal
-- ----------------------------------------------------------------------------
-- Un catalog es el contenedor de mas alto nivel en Unity Catalog.
-- Pensalo como una "base de datos de bases de datos".
CREATE CATALOG IF NOT EXISTS combustibles_hn
  COMMENT 'Proyecto personal: precios semanales de combustibles en Honduras (2017-actual).';

-- Posicionarse en el catalog recien creado.
USE CATALOG combustibles_hn;


-- ----------------------------------------------------------------------------
-- PASO 2: Crear los ESQUEMAS (las 3 capas del medallion)
-- ----------------------------------------------------------------------------
-- Un schema agrupa tablas relacionadas. En medallion tenemos uno por capa.

CREATE SCHEMA IF NOT EXISTS bronze
  COMMENT 'BRONZE: datos crudos tal cual llegan (CSV historico + scraping proceso.hn).';

CREATE SCHEMA IF NOT EXISTS silver
  COMMENT 'SILVER: datos limpios, con fechas parseadas y tipos correctos. Una sola tabla unificada.';

CREATE SCHEMA IF NOT EXISTS gold
  COMMENT 'GOLD: modelo analitico listo para BI (hechos, dimensiones, agregados).';


-- ----------------------------------------------------------------------------
-- PASO 3: Crear el VOLUME para archivos raw
-- ----------------------------------------------------------------------------
-- Un volume es una "carpeta" dentro de Unity Catalog para guardar archivos
-- (CSV, parquet, imagenes, HTML). Reemplaza al viejo DBFS.

CREATE VOLUME IF NOT EXISTS bronze.raw
  COMMENT 'Archivos crudos: CSV historico, HTML descargado de proceso.hn (si falla scraping).';


-- ----------------------------------------------------------------------------
-- PASO 4: Verificar la estructura creada
-- ----------------------------------------------------------------------------
-- Estos SELECT te muestran que todo quedo bien:

SHOW CATALOGS LIKE 'combustibles_hn';
-- Resultado esperado: 1 fila con 'combustibles_hn'.

SHOW SCHEMAS IN combustibles_hn;
-- Resultado esperado: 3 filas: bronze, silver, gold (mas information_schema).

SHOW VOLUMES IN combustibles_hn.bronze;
-- Resultado esperado: 1 fila con 'raw'.


-- ----------------------------------------------------------------------------
-- PASO 5: (OPCIONAL) Crear tablas vacias para reservar nombres
-- ----------------------------------------------------------------------------
-- Esto NO es obligatorio. Las tablas se crean automaticamente cuando corras
-- los notebooks 01, 02 y 03 con saveAsTable() o CREATE TABLE AS SELECT.
-- Pero si queres ver la estructura completa desde el inicio, descomenta:


-- Bronze (4 tablas)
CREATE TABLE IF NOT EXISTS combustibles_hn.bronze.bronze_csv_historico (placeholder STRING)
  COMMENT 'CSV historico 2017-2024, AS-IS desde el volumen raw.';
CREATE TABLE IF NOT EXISTS combustibles_hn.bronze.bronze_web_2024 (placeholder STRING)
  COMMENT 'Scraping proceso.hn ano 2024.';
CREATE TABLE IF NOT EXISTS combustibles_hn.bronze.bronze_web_2025 (placeholder STRING)
  COMMENT 'Scraping proceso.hn ano 2025.';
CREATE TABLE IF NOT EXISTS combustibles_hn.bronze.bronze_web_2026 (placeholder STRING)
  COMMENT 'Scraping proceso.hn ano 2026.';

-- Silver (1 tabla)
CREATE TABLE IF NOT EXISTS combustibles_hn.silver.silver_precios_semanales (placeholder STRING)
  COMMENT 'Precios semanales unificados, limpios, tipados.';

-- Gold (7 tablas + 1 vista)
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_calendario (placeholder STRING)
  COMMENT 'Dimension calendario.';
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_fact_precios (placeholder STRING)
  COMMENT 'Hecho: precios unpivot por combustible y semana.';
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_super (placeholder STRING);
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_regular (placeholder STRING);
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_diesel (placeholder STRING);
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_kerosene (placeholder STRING);
CREATE TABLE IF NOT EXISTS combustibles_hn.gold.gold_tendencias (placeholder STRING);



-- ----------------------------------------------------------------------------
-- PASO 6: Permisos (OPCIONAL, solo si vas a compartir el catalogo)
-- ----------------------------------------------------------------------------
-- En Free Edition vos sos el unico usuario, asi que NO necesitas correr esto.
-- Lo dejo de referencia por si despues invitas colaboradores:

/*
GRANT USE CATALOG ON CATALOG combustibles_hn TO `usuario@correo.com`;
GRANT USE SCHEMA  ON SCHEMA  combustibles_hn.gold TO `usuario@correo.com`;
GRANT SELECT      ON SCHEMA  combustibles_hn.gold TO `usuario@correo.com`;
*/


-- ============================================================================
-- SIGUIENTE PASO:
--   1. Sube tu archivo CSV historico al volumen:
--      Catalog Explorer -> combustibles_hn -> bronze -> raw -> Upload
--      Renombralo a: historico_2017_2024.csv
--   2. Importa los 3 notebooks en tu Workspace:
--      01_bronze_ingesta.py
--      02_silver_limpieza.py
--      03_gold_modelo.py
--   3. Corre los 3 notebooks en orden.
-- ============================================================================
